# GPU RAID — воркер на Kaggle (2×T4 = два воркера с одной сессии)

**Перед запуском (панель справа → Session options):**
1. Accelerator: **GPU T4 x2**
2. Internet: **On** (нужна верификация телефона в настройках аккаунта)
3. Модели: подключите свои Kaggle Datasets с моделями через **Add Input**
   (см. `kaggle_dataset_prep.ipynb`) — старт за ~3 минуты без скачивания.

Лимиты Kaggle: ~30 ч GPU в неделю, сессия до 12 ч. Держите вкладку открытой.

Запустите ячейки сверху вниз. В конце будут напечатаны **две строки** —
вставьте их в панель GPU RAID на вашем ПК (Добавить воркеров).

In [ ]:
# ================= КОНФИГ =================
REPO_URL = "https://github.com/Weloyo/ComfyUI-GPU-RAID"  # репозиторий расширения
TOKEN = ""             # пусто = сгенерировать автоматически
MODEL_PRESET = "none"  # "sdxl" | "minimax_h3" | "none" (none = только Datasets)
USE_DATASETS = True    # линковать модели из /kaggle/input/*

# 2xT4 = два воркера. Для ТЯЖЁЛЫХ моделей (minimax_h3: ~40 ГБ весов при 32 ГБ RAM
# сессии) авто-переключаемся на один инстанс на одной T4 — иначе не хватит памяти.
# Хотите оставить два инстанса всё равно — переопределите DUAL_T4 = True после этой строки.
DUAL_T4 = MODEL_PRESET != "minimax_h3"
EXTRA_ARGS = ("--force-fp16",)  # T4 без bf16
if MODEL_PRESET == "minimax_h3":
    EXTRA_ARGS = EXTRA_ARGS + ("--cache-none",)  # экономим системную RAM на тяжёлой модели

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print(f"config ok: DUAL_T4={DUAL_T4} EXTRA_ARGS={EXTRA_ARGS}")

In [ ]:
# ============ ИСХОДНИКИ РАСШИРЕНИЯ ============
import os, sys, subprocess
SRC = "/kaggle/working/gpu-raid-src"
if not os.path.isdir(SRC):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, SRC])
    if r.returncode != 0:
        # fallback: исходники, приложенные как Kaggle Dataset (папка, где есть __init__.py и gpu_raid/)
        import glob, shutil
        cands = [p for p in glob.glob("/kaggle/input/*/gpu_raid") if os.path.isdir(p)]
        assert cands, "git clone не удался и датасет с исходниками не найден"
        shutil.copytree(os.path.dirname(cands[0]), SRC)
sys.path.insert(0, os.path.join(SRC, "scripts"))
import worker_bootstrap as wb
TOKEN = wb.gen_token(TOKEN)
print("TOKEN:", TOKEN)
subprocess.run(["nvidia-smi", "-L"])

In [ ]:
# ============ УСТАНОВКА + ЗАПУСК + ТУННЕЛИ ============
# DUAL_T4=True: два инстанса ComfyUI (по одному на T4) и два туннеля.
# DUAL_T4=False: один инстанс — для тяжёлых моделей (minimax_h3).
info = wb.bring_up(
    gpuraid_src=SRC,
    comfy_dir="/kaggle/working/ComfyUI",
    token=TOKEN,
    gpus=(0, 1) if DUAL_T4 else (0,),
    base_port=8188,
    extra_args=EXTRA_ARGS,
    use_datasets=USE_DATASETS,
    hf_preset=MODEL_PRESET,
    hf_token=HF_TOKEN,
    name_prefix="kaggle",
)

In [ ]:
# ============ KEEPALIVE ============
# Держит сессию живой и показывает хвосты логов. Остановить: кнопка Stop.
wb.keepalive(info["procs"], log_paths=["/tmp/comfy_8188.log", "/tmp/comfy_8189.log"])

### Если что-то пошло не так

- **Туннель умер** (воркер покраснел в панели): перезапустите ячейку
  «УСТАНОВКА + ЗАПУСК» — она идемпотентна; затем в панели GPU RAID нажмите
  **URL** у воркера и вставьте новый адрес (токен и remap сохранятся).
- **cloudflared не выдаёт URL**: запустите в новой ячейке pinggy
  (`!ssh -o StrictHostKeyChecking=no -p 80 -R0:localhost:8188 a.pinggy.io`)
  и используйте его https-URL. У бесплатного pinggy URL меняется каждые 60 минут.
- **Нет моделей**: проверьте, что Dataset подключён через Add Input, а внутри
  него есть подпапки `checkpoints/`, `vae/`, … или `manifest.json`.
- Лог инстанса: `!tail -50 /tmp/comfy_8188.log`